In [1]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report
import rtdl

from src.preprocessor import prepare_give_me_some_credit_grandmaster

file_path = '../data/raw/Give Me Some Credit/cs-training.csv'

In [2]:
X_train, X_test, y_train, y_test, feature_names = prepare_give_me_some_credit_grandmaster(file_path)

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_clean = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_clean = scaler.transform(imputer.transform(X_test))

X_train_tensor = torch.tensor(X_train_clean, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_clean, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = rtdl.FTTransformer.make_default(
    n_num_features=X_train_tensor.shape[1],
    cat_cardinalities=None,
    last_layer_query_idx=[-1],
    d_out=1,
)
model.to(device)

imbalance_ratio = (len(y_train) - sum(y_train)) / sum(y_train)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([imbalance_ratio]).to(device))
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

model.train()
for epoch in range(3):
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x, x_cat=None)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor.to(device), x_cat=None)
    test_probs = torch.sigmoid(test_preds).cpu().numpy().flatten()

auc = roc_auc_score(y_test, test_probs)

best_threshold = 0.50
best_f1 = 0.0

from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_test, test_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred_optimal = (test_probs >= best_threshold).astype(int)

print(f"--- FT-Transformer Performance ---")
print(f"Device used: {device}")
print(f"AUC Score: {auc:.4f}")
print(f"\n--- Classification Report @ Threshold {best_threshold:.4f} ---")
print(classification_report(y_test, y_pred_optimal))

--- FT-Transformer Performance ---
Device used: cpu
AUC Score: 0.8371

--- Classification Report @ Threshold 0.8448 ---
              precision    recall  f1-score   support

           0       0.96      0.95      0.96     27995
           1       0.41      0.43      0.42      2005

    accuracy                           0.92     30000
   macro avg       0.68      0.69      0.69     30000
weighted avg       0.92      0.92      0.92     30000

